# Conversational Agent

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) 
openai.api_key = os.environ['OPENAI_API_KEY']

First we will implement the steps manually then we will wrap it inside a loop

In [2]:
from langchain.tools import tool

In [3]:
#tool1: get the current temperature

import requests
from pydantic import BaseModel, Field
import datetime

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""
    
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']
    
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]
    
    return f'The current temperature is {current_temperature}°C'

In [ ]:
#tool2 get info from wikipedia

import wikipedia

@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            wikipedia.exceptions.PageError, 
            wikipedia.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)


In [5]:
#define a list of tools

tools = [get_current_temperature, search_wikipedia]

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain.tools.render import format_tool_to_openai_function

MessagesPlaceholder holds the messages of the intermediate steps

In [7]:
#define the prompt

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an helpful assistant"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name = "agent_scratchpad")
    ]
)

In [8]:
#formatting the tools to openai functions

functions = [format_tool_to_openai_function(x) for x in tools]

In [9]:
#define the model

model = ChatOpenAI(temperature = 0).bind(functions = functions)

In [10]:
#define the chain

chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [11]:
result1 = chain.invoke({"input": "what is the weather in Delhi, India", "agent_scratchpad": []})

In [12]:
result1

AgentActionMessageLog(tool='get_current_temperature', tool_input={'latitude': 28.7041, 'longitude': 77.1025}, log="\nInvoking: `get_current_temperature` with `{'latitude': 28.7041, 'longitude': 77.1025}`\n\n\n", message_log=[AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_temperature', 'arguments': '{"latitude":28.7041,"longitude":77.1025}'}})])

### The new `.log` attribute

Compared to before, `result1` (an `AgentActionMessageLog`) now also has a `.log` attribute - a plain-text, human-readable description of what the agent decided to do (e.g. `"Invoking: get_current_temperature with {...}"`). It's separate from `.tool` and `.tool_input`, which are the structured values you'd actually use in code - `.log` is more for humans reading/debugging what the agent is doing, not something you'd typically parse programmatically.


In [13]:
result1.log

"\nInvoking: `get_current_temperature` with `{'latitude': 28.7041, 'longitude': 77.1025}`\n\n\n"

In [14]:
#lets call the tool

observation = get_current_temperature(result1.tool_input)

In [15]:
observation

'The current temperature is 27.4°C'

In [16]:
type(observation)

str

we called the llm then we called the tool as per llm's request, inorder to wrap it inside a loop we need something to hold up the intermediate results i.e. llm calls tool then tool responds we want to hold the user message llm's tool calling and the tool's response then we will give this all to the llm and it can decide and respond us back, for this we will use scratchpad

In [17]:
#note that the library imported is format_to_openai_functions with no tool in between

from langchain.agents.format_scratchpad import format_to_openai_functions

### What is the agent scratchpad?

When an agent runs, it often loops: it decides to call a tool, gets back an observation (the tool's result), and needs to pass that short history back into the LLM so it knows what already happened and can either respond or decide to call another tool. This running history of intermediate steps is the "scratchpad".

### Why format it this way?

OpenAI's API expects a specific message sequence when passing tool-usage history back to the model:
* an `AIMessage` containing the `function_call` (the agent deciding to use a tool, and with what arguments)
* a `FunctionMessage` containing the actual result the tool returned

`format_to_openai_functions` takes a list of `(action, observation)` tuples - here just `(result1, observation)` - and converts each pair into that exact `[AIMessage, FunctionMessage]` format. The resulting list gets passed into `chain.invoke(...)` as `agent_scratchpad`, so the LLM can read the tool's result and answer the original question.


In [18]:
format_to_openai_functions([(result1, observation), ]) #converts our (agent_action, tool_result) pair into the [AIMessage, FunctionMessage] format the LLM expects to see


[AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_temperature', 'arguments': '{"latitude":28.7041,"longitude":77.1025}'}}),
 FunctionMessage(content='The current temperature is 27.4°C', name='get_current_temperature')]

you can see two messages aimessage as well as functionmesage in the list which is great, so we will pass on this list to llm to get a final response

In [19]:
result2 = chain.invoke({"input": "what is the weather in Delhi, India", "agent_scratchpad": format_to_openai_functions([(result1, observation), ])})

In [20]:
result2

AgentFinish(return_values={'output': 'The current temperature in Delhi, India is 27.4°C.'}, log='The current temperature in Delhi, India is 27.4°C.')

This is the final response from the llm

so we just need to wrap these manual steps in a loop

## What were we doing manually?

1. Calling `format_to_openai_functions` to build the scratchpad
2. Passing in the user's prompt
3. Getting the function call decision from the LLM
4. Calling that function with the arguments the LLM gave us
5. Giving the function's output back to the model (as part of the scratchpad)
6. Getting the LLM's final response


In [28]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.agent import AgentFinish

### What does `RunnablePassthrough` do, and why use it here?

`RunnablePassthrough.assign(...)` takes whatever input dict comes in, keeps every existing key as-is (passes it through unchanged), and adds new key(s) computed from that same input - here, it adds `agent_scratchpad` by running `format_to_openai_functions` on `intermediate_steps`.

We need it because our `prompt` expects both `input` and `agent_scratchpad` as variables, but the loop only naturally has `input` and `intermediate_steps` (the running list of past tool calls). `RunnablePassthrough.assign` bridges that gap - it lets us compute `agent_scratchpad` from `intermediate_steps` on the fly, right inside the chain, instead of having to manually call `format_to_openai_functions` ourselves before every single `invoke()` (which is exactly what we were doing by hand in the cells above).


In [22]:
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad = lambda x: format_to_openai_functions(x["intermediate_steps"])
) | chain

In [26]:
def run_agent(user_input):
    intermediate_steps = []
    while True:
        result = agent_chain.invoke({
            "input": user_input,
            "intermediate_steps": intermediate_steps
        })
        
        if isinstance(result, AgentFinish):
            return result
        
        tool = {
            "search_wikipedia": search_wikipedia,
            "get_current_temperature": get_current_temperature
        }[result.tool] #either this or you can do it in a naive way
        
        observation = tool.run(result.tool_input)
        intermediate_steps.append((result, observation))

In [29]:
run_agent("what is the current temperature in New Delhi, India")

AgentFinish(return_values={'output': 'The current temperature in New Delhi, India is 27.1°C.'}, log='The current temperature in New Delhi, India is 27.1°C.')

In [30]:
run_agent("hello")

AgentFinish(return_values={'output': 'Hello! How can I assist you today?'}, log='Hello! How can I assist you today?')

## Better way to do this

In [36]:
tools = [search_wikipedia, get_current_temperature]

In [39]:
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(agent = agent_chain, tools = tools, verbose = True)

In [40]:
agent_executor.invoke({"input": "summarize what is langchain"})



> Entering new AgentExecutor chain...

Invoking: `search_wikipedia` with `{'query': 'Langchain'}`


Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: List of artificial intelligence companies
Summary: Below is a list of notable companies that primarily focus on artificial intelligence (AI). Companies that simply make use of AI but have a different primary focus are not included.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given inp

{'input': 'summarize what is langchain',
 'output': 'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. It is used for tasks such as document analysis and summarization, chatbots, and code analysis.'}